<a href="https://colab.research.google.com/github/marcelovolta/sr_uba_2026/blob/main/notebooks%20profesor/11_Dos_torres_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -qU keras-rs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 4.6 MB/s eta 0:00:00


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras_rs
import keras
import tensorflow as tf
import sqlite3
import pandas as pd

# Datos

In [ ]:
conn = sqlite3.connect('data.db')

df_ratings = pd.read_sql("SELECT id_lector, id_libro, rating FROM interacciones", conn)
df_ratings.head()

,id_lector,id_libro,rating
0,bioaqua,las-puertas-de-anubis,4
1,bioaqua,antonio-b-el-ruso-ciudadano-de-tercera,10
2,bioaqua,la-momia,10
3,bioaqua,el-clan-del-oso-cavernario-los-hijos-de-la-tie...,10
4,bioaqua,dime-quien-soy-1,10


In [ ]:
# diccionarios de traducción
unique_user_id = df_ratings["id_lector"].unique().tolist()
user_id_2_id_mapping = {t: i for i, t in enumerate(unique_user_id)}
id_2_user_user_mapping = {user_id_2_id_mapping[k]: k for k in user_id_2_id_mapping}

unique_item_id = df_ratings["id_libro"].unique().tolist()
item_id_2_id_mapping = {t: i for i, t in enumerate(unique_item_id)}
id_2_item_id_mapping = {item_id_2_id_mapping[k]: k for k in item_id_2_id_mapping}

In [ ]:
df_ratings["id_lector"] = df_ratings["id_lector"].apply(lambda x: user_id_2_id_mapping[x])
df_ratings["id_libro"] = df_ratings["id_libro"].apply(lambda x: item_id_2_id_mapping[x])

df_ratings.head()

,id_lector,id_libro,rating
0,0,0,4
1,0,1,10
2,0,2,10
3,0,3,10
4,0,4,10


In [ ]:
dataset_inputs = {
    "user_id": df_ratings['id_lector'].to_numpy(dtype="float32"),
    'item_id': df_ratings['id_libro'].to_numpy(dtype="float32")
}

dataset_outputs = (df_ratings['rating'].to_numpy() - 1.0) / 9.0

dataset = tf.data.Dataset.from_tensor_slices((dataset_inputs, dataset_outputs))
dataset = dataset.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

cut = int(df_ratings.shape[0] * 0.8)
train_dataset = dataset.take(cut).batch(256).cache()
test_dataset = dataset.skip(df_ratings.shape[0] - cut).batch(256).cache()

## Modelo

In [ ]:
class RankingModel(keras.Model):
    """Crea un modelo de ranking

    Args:
      num_users: Number of entries in the user embedding table.
      num_items: Number of entries in the candidate embedding table.
      embedding_dimension: Output dimension for user and movie embedding tables.
    """

    def __init__(
        self,
        num_users,
        num_items,
        embedding_dimension=512, # embedding_dimension es hiperparámetro
        **kwargs,
    ):
        super().__init__(**kwargs)

        # Embedding de los usuarios
        self.user_embedding = keras.layers.Embedding(num_users, embedding_dimension)

        # Embedding de los items
        self.item_embedding = keras.layers.Embedding(num_items, embedding_dimension)

        # Predicciones
        self.ratings = keras.Sequential(
            [
                # Capas densas
                keras.layers.Dense(256, activation="relu"),
                keras.layers.Dense(64, activation="relu"),

                # Predicciones en la capa final
                keras.layers.Dense(1, activation="sigmoid"),
            ]
        )

    def call(self, inputs):
        user_id = inputs["user_id"]
        item_id = inputs["item_id"]

        user_embeddings = self.user_embedding(user_id)
        item_embeddings = self.item_embedding(item_id)
        return self.ratings(keras.ops.concatenate([user_embeddings, item_embeddings], axis=1))

## Entrenamiento y evaluación

In [ ]:
ranking_model = RankingModel(len(unique_user_id) + 1, len(unique_item_id) + 1)
ranking_model.compile(
    loss=keras.losses.MeanSquaredError(),
    metrics=[keras.metrics.RootMeanSquaredError()],
    optimizer=keras.optimizers.Adagrad(learning_rate=0.1),
)

In [ ]:
history = ranking_model.fit(train_dataset, validation_data=test_dataset, validation_freq=5, epochs=10, batch_size=256)

/usr/local/lib/python3.13/dist-packages/jax/_src/core.py:1962: UserWarning: Explicitly requested dtype float64 requested in ShapeDtypeStruct is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return aval_fn(x)


Epoch 1/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0378 - root_mean_squared_error: 0.1945
Epoch 2/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0337 - root_mean_squared_error: 0.1835
Epoch 3/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0320 - root_mean_squared_error: 0.1790
Epoch 4/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0310 - root_mean_squared_error: 0.1761
Epoch 5/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - loss: 0.0303 - root_mean_squared_error: 0.1739 - val_loss: 0.0315 - val_root_mean_squared_error: 0.1774
Epoch 6/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0296 - root_mean_squared_error: 0.1721
Epoch 7/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0291 - root_mean_squared_error: 0.1705
Epoch 8/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0285 - root_mean_squared_error: 0.1689
Epoch 9/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0280 - root_mean_squared_error: 0.1674
Epoch 10/10
1442

## Predicción

In [ ]:
#user_id_2_id_mapping["popocito"]
#item_id_2_id_mapping["la-catedral-del-mar"]

123

In [ ]:
user_id = user_id_2_id_mapping["popocito"]
item_ids = [item_id_2_id_mapping[i] for i in ["1984", "el-hobbit", "el-silmarillion", "la-catedral-del-mar"]]

predictions = ranking_model.predict({
    "user_id": keras.ops.array([user_id] * len(item_ids)),
    "item_id": keras.ops.array(item_ids),
})

predictions = keras.ops.convert_to_numpy(keras.ops.squeeze(predictions, axis=1))

for id_libro, prediction in zip(item_ids, predictions):
    print(f"{id_2_item_id_mapping[id_libro]} --> {prediction * 9.0 + 1.0:,.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
1984 --> 7.54
el-hobbit --> 7.27
el-silmarillion --> 6.15
la-catedral-del-mar --> 6.43


In [ ]:
user_id = 42

df_item_ids = pd.read_sql("SELECT id_libro FROM libros WHERE id_libro NOT IN (SELECT id_libro FROM interacciones WHERE id_lector = ?)", conn, params=[user_id])
item_ids = [item_id_2_id_mapping[u] for u in df_item_ids["id_libro"].to_list() if item_id_2_id_mapping.get(u)]

print(len(item_ids))

predictions = ranking_model.predict({
    "user_id": keras.ops.array([user_id] * len(item_ids)),
    "item_id": keras.ops.array(item_ids),
})

48066
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
